# GeoGen AI — train the EuroSAT conditional GAN

This notebook trains the same `models.Generator` that `app.py` loads. It saves the checkpoint in the project folder and checks it by generating one PNG. Run cells from top to bottom.

**Before starting:** Upload the project source folder to Google Drive as `MyDrive/GeoGen-AI` (do not upload `.venv`). In Colab choose **Runtime → Change runtime type → T4 GPU** or another GPU. Training downloads EuroSAT RGB (about 95 MB) and can take time; the latest checkpoint is saved after each epoch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/GeoGen-AI')
required = ['models.py', 'train_gan.py', 'dataset_utils.py', 'app.py', 'requirements.txt']
missing = [name for name in required if not (PROJECT / name).is_file()]
if missing:
    raise FileNotFoundError(f'Project folder is missing: {missing}. Upload GeoGen-AI to MyDrive first.')
%cd $PROJECT

## Install the project dependencies

In [ ]:
%pip install -r requirements.txt

## Confirm Colab sees a GPU
Stop here and choose a GPU runtime if this check fails. The laptop's CPU is not used by this notebook.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No Colab GPU detected. Select Runtime > Change runtime type > GPU, reconnect, and rerun.')

## Download and prepare EuroSAT RGB
This downloads the official RGB archive, verifies its published checksum, then extracts the 10 class folders.

In [ ]:
!python dataset_utils.py

## Train and save the model
The default 50 epochs are for a full run; loss values are training diagnostics, not proof of image quality. Reduce `EPOCHS` for a quicker pipeline trial, then rerun longer if time permits. The script saves `artifacts/checkpoints/conditional_dcgan_latest.pt` after each completed epoch.

In [ ]:
EPOCHS = 50
BATCH_SIZE = 64
!python train_gan.py --epochs $EPOCHS --batch-size $BATCH_SIZE --workers 2

## Verify the app-compatible checkpoint and create one PNG
This cell instantiates `Generator` from `models.py`, loads the just-trained weights and exports one generated AnnualCrop sample. It does not use a stock or placeholder image.

In [ ]:
from torchvision.utils import save_image
from models import Generator
CHECKPOINT_PATH = PROJECT / 'artifacts' / 'checkpoints' / 'conditional_dcgan_latest.pt'
payload = torch.load(CHECKPOINT_PATH, map_location='cuda', weights_only=False)
generator = Generator(latent_dim=payload['latent_dim']).cuda()
generator.load_state_dict(payload['generator'])
generator.eval()
class_name = 'AnnualCrop'
class_id = payload['class_names'].index(class_name)
with torch.no_grad():
    label = torch.tensor([class_id], device='cuda')
    noise = torch.randn(1, payload['latent_dim'], device='cuda')
    generated = generator(noise, label).cpu()
SAMPLE_PATH = PROJECT / 'artifacts' / 'generated' / 'colab_AnnualCrop_sample.png'
SAMPLE_PATH.parent.mkdir(parents=True, exist_ok=True)
save_image(((generated[0] + 1) / 2).clamp(0, 1), SAMPLE_PATH)
print('Checkpoint:', CHECKPOINT_PATH)
print('Generated image:', SAMPLE_PATH)

## Download the checkpoint and sample to your laptop
The checkpoint is saved in Drive already. These downloads copy it to your computer; move the checkpoint into your local project's `artifacts/checkpoints/` folder. Then run `streamlit run app.py` and use the app's Download button for a PNG.

In [ ]:
from google.colab import files
files.download(str(CHECKPOINT_PATH))
files.download(str(SAMPLE_PATH))